## 2.Inspect

In [1]:
import pandas as pd

MY_ID = 3246

raw = pd.read_csv("data/almaty_apartments_raw.csv")
ids = raw["listing_id"].drop_duplicates().sample(frac=0.9, random_state=MY_ID)
df = raw[raw["listing_id"].isin(ids)].reset_index(drop=True)

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe(include="all").T)

(1155, 11)
listing_id           object
listed_date          object
district             object
rooms                object
area_m2              object
floor                 int64
total_floors          int64
year_built          float64
building_type        object
ceiling_height_m    float64
price_kzt            object
dtype: object
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                 0
total_floors          0
year_built           57
building_type         0
ceiling_height_m    139
price_kzt             0
dtype: int64
                   count unique         top freq        mean         std  min  \
listing_id          1155   1080   ALM-10932    3         NaN         NaN  NaN   
listed_date         1155    247  2026-04-21   10         NaN         NaN  NaN   
district            1155     48       Medeu  148         NaN         NaN  NaN   
rooms               1155      6           2  431         NaN      

In [2]:
print("rooms уникальные:", df["rooms"].unique())
print()
print("area_m2 примеры:", df["area_m2"].unique()[:20])
print()
print("price_kzt примеры:", df["price_kzt"].unique()[:20])
print()
print("district уникальные (все 48):")
print(sorted(df["district"].unique()))

rooms уникальные: ['4' '2' '3' '1' '5' 'studio']

area_m2 примеры: ['90.8' '52.1' '69.8' '82.8' '56.2' '64.3' '116.1' '56.9' '48.6' '36.6'
 '53.5' '67.6' '133.0' '59.0' '48.2' '56.3' '67.2' '110.2' '57.0' '35.4']

price_kzt примеры: ['41,560,000' '27700000' '78490000' '95690000' '30040000' '59000000'
 '63730000' '25720000' '26120000' '29170000' '29050000' '43390000'
 '177470000' '70470000' '38740000' '29 060 000 ₸' '27670000' '60140000'
 '25760000' '23580000']

district уникальные (все 48):
[' Alatau ', ' Almaly ', ' Auezov ', ' Bostandyk ', ' Medeu ', ' Nauryzbay ', ' Turksib ', ' Zhetysu ', 'ALATAU', 'ALMALY', 'AUEZOV', 'Alatau', 'Alatauskiy', 'Almalinskiy', 'Almaly', 'Auezov', 'Auezovskiy', 'BOSTANDYK', 'Bostandyk', 'Bostandykskiy', 'MEDEU', 'Medeu', 'Medeuskiy', 'NAURYZBAY', 'Nauryzbay', 'Nauryzbayskiy', 'TURKSIB', 'Turksib', 'Turksibskiy', 'ZHETYSU', 'Zhetysu', 'Zhetysuskiy', 'alatau', 'almaly', 'auezov', 'bostandyk', 'medeu', 'nauryzbay', 'turksib', 'zhetysu', 'Алатауский', 'Алма

## Data Inspection Findings

- `rooms`, `area_m2`, `price_kzt` are stored as `object` instead of numeric types.
  - `rooms` contains the string `"studio"` alongside numeric strings.
  - `price_kzt` has three formats: plain digits, comma-separated, and space-separated with a `₸` symbol.
- `listing_id`: 1155 rows but only 1080 unique values → duplicates/reposts present.
- `floor` has a minimum of -1 (impossible value, likely a sentinel).
- `year_built` has a minimum of 0 (impossible, likely a sentinel) and 57 missing values.
- `ceiling_height_m` has 139 missing values.
- `district` has 48 raw string variants (casing, whitespace, transliteration suffix `-skiy`, Russian names) that map down to 8 canonical districts: Alatau, Almaly, Auezov, Bostandyk, Medeu, Nauryzbay, Turksib, Zhetysu.

In [3]:
validation_spec = {
    "listing_id":       {"dtype": "string", "notes": "unique per real listing, but duplicates/reposts exist"},
    "listed_date":      {"dtype": "datetime"},
    "district":         {"dtype": "category", "allowed": ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]},
    "rooms":            {"dtype": "int", "min": 0, "max": 10, "notes": "'studio' -> 0"},
    "area_m2":          {"dtype": "float", "min": 10, "max": 500},
    "floor":            {"dtype": "int", "min": 1, "max": 60, "notes": "current min=-1 is impossible"},
    "total_floors":     {"dtype": "int", "min": 1, "max": 60},
    "year_built":       {"dtype": "int", "min": 1950, "max": 2026, "notes": "current min=0 is impossible/sentinel"},
    "building_type":    {"dtype": "category"},
    "ceiling_height_m": {"dtype": "float", "min": 2.0, "max": 4.5},
    "price_kzt":        {"dtype": "float", "min": 1_000_000, "max": 500_000_000},
}
validation_spec

{'listing_id': {'dtype': 'string',
  'notes': 'unique per real listing, but duplicates/reposts exist'},
 'listed_date': {'dtype': 'datetime'},
 'district': {'dtype': 'category',
  'allowed': ['Alatau',
   'Almaly',
   'Auezov',
   'Bostandyk',
   'Medeu',
   'Nauryzbay',
   'Turksib',
   'Zhetysu']},
 'rooms': {'dtype': 'int', 'min': 0, 'max': 10, 'notes': "'studio' -> 0"},
 'area_m2': {'dtype': 'float', 'min': 10, 'max': 500},
 'floor': {'dtype': 'int',
  'min': 1,
  'max': 60,
  'notes': 'current min=-1 is impossible'},
 'total_floors': {'dtype': 'int', 'min': 1, 'max': 60},
 'year_built': {'dtype': 'int',
  'min': 1950,
  'max': 2026,
  'notes': 'current min=0 is impossible/sentinel'},
 'building_type': {'dtype': 'category'},
 'ceiling_height_m': {'dtype': 'float', 'min': 2.0, 'max': 4.5},
 'price_kzt': {'dtype': 'float', 'min': 1000000, 'max': 500000000}}

## 3.Clean

In [4]:
dup_ids = df["listing_id"][df["listing_id"].duplicated(keep=False)]
dup_rows = df[df["listing_id"].isin(dup_ids)].sort_values("listing_id")
print(dup_rows.shape)
dup_rows.head(20)

(147, 11)


,listing_id,listed_date,district,rooms,area_m2,floor,total_floors,year_built,building_type,ceiling_height_m,price_kzt
8,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
962,ALM-10003,2026-05-04,Auezov,2,48.6,2,5,1978.0,panel,2.55,28320000
819,ALM-10003,2026-05-29,Auezov,2,48.6,2,5,1978.0,panel,2.55,26120000
935,ALM-10009,2025-12-03,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,42700000
179,ALM-10009,2026-01-19,Medeuskiy,1,33.4,2,9,1992.0,brick,3.19,39400000
453,ALM-10016,2026-01-29,Auezov,1,39.7,13,25,2022.0,monolith,2.80,32100000
477,ALM-10016,2026-01-20,Auezov,1,39.7,13,25,2022.0,monolith,2.80,35130000
255,ALM-10021,2026-04-06,Alatau,3,71.3,9,12,1988.0,panel,2.69,32300000
609,ALM-10021,2026-05-09,Alatau,3,71.3,9,12,1988.0,panel,2.69,30090000
52,ALM-10056,2026-02-08,Bostandyk,4,107.2,9,9,2006.0,monolith,2.81,101060000


In [5]:
before = len(df)

df["listed_date"] = pd.to_datetime(df["listed_date"])

df_clean = (
    df.sort_values("listed_date")
      .drop_duplicates(subset="listing_id", keep="last")
      .reset_index(drop=True)
)

after = len(df_clean)
print(f"Rows before: {before}, after dedup: {after}, removed: {before - after}")

Rows before: 1155, after dedup: 1080, removed: 75


In [6]:
canonical_districts = ["Alatau", "Almaly", "Auezov", "Bostandyk", "Medeu", "Nauryzbay", "Turksib", "Zhetysu"]

ru_map = {
    "Алатауский": "Alatau",
    "Алмалинский": "Almaly",
    "Ауэзовский": "Auezov",
    "Бостандыкский": "Bostandyk",
    "Медеуский": "Medeu",
    "Наурызбайский": "Nauryzbay",
    "Турксибский": "Turksib",
    "Жетысуский": "Zhetysu",
}

def clean_district(x):
    x = x.strip()
    if x in ru_map:
        return ru_map[x]
    x = x.lower()
    for suffix in ["inskiy", "skiy"]:
        if x.endswith(suffix):
            x = x[: -len(suffix)]
            break
    for d in canonical_districts:
        if d.lower().startswith(x):
            return d
    return None

df_clean["district"] = df_clean["district"].apply(clean_district)

print(df_clean["district"].unique())
print(df_clean["district"].isna().sum())

['Almaly' 'Zhetysu' 'Alatau' 'Medeu' 'Bostandyk' 'Auezov' 'Turksib'
 'Nauryzbay']
0


Districts normalized to 8 canonical values by stripping whitespace,lowercasing,mapping Russian names via a dictionary and stripping transliteration suffixes (`-skiy`/`-inskiy`). `Almalinskiy` required a `startswith` match rather than exact match due to an irregular suffix pattern

In [7]:
# rooms: 'studio' в 1,остальное в число
df_clean["rooms"] = df_clean["rooms"].replace("studio", "1").astype(int)

# area_m2: убрать "m²",меняем запятую на точку,приведем к float
df_clean["area_m2"] = (
    df_clean["area_m2"]
    .astype(str)
    .str.replace("m²", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
    .astype(float)
)

# price_kzt: убираем запятые,пробелы,символ ₸
df_clean["price_kzt"] = (
    df_clean["price_kzt"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("₸", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.strip()
    .astype(float)
)

print(df_clean[["rooms", "area_m2", "price_kzt"]].dtypes)
print(df_clean[["rooms", "area_m2", "price_kzt"]].describe())

rooms          int64
area_m2      float64
price_kzt    float64
dtype: object
             rooms      area_m2     price_kzt
count  1080.000000  1080.000000  1.080000e+03
mean      2.261111    68.907963  4.631151e+07
std       1.008934    59.780981  2.521701e+07
min       1.000000    25.300000  1.144000e+07
25%       1.000000    45.000000  2.828750e+07
50%       2.000000    58.750000  4.019000e+07
75%       3.000000    78.650000  5.732500e+07
max       5.000000   879.000000  1.930000e+08


## 4.Sentinels & outliers

In [8]:
print("floor value counts (низкие и высокие):")
print(df_clean["floor"].value_counts().sort_index().head(5))
print()
print("year_built value counts (низкие):")
print(df_clean["year_built"].value_counts().sort_index().head(10))
print()
print("ceiling_height_m value counts (низкие):")
print(df_clean["ceiling_height_m"].value_counts().sort_index().head(10))

floor value counts (низкие и высокие):
floor
-1     11
 1    141
 2    124
 3    152
 4    109
Name: count, dtype: int64

year_built value counts (низкие):
year_built
0.0       20
1950.0     3
1951.0     2
1952.0     6
1953.0     5
1954.0     3
1955.0     3
1956.0     2
1957.0     4
1958.0     6
Name: count, dtype: int64

ceiling_height_m value counts (низкие):
ceiling_height_m
0.00    21
2.50     7
2.51    24
2.52    19
2.53    26
2.54    11
2.55    21
2.56    14
2.57    14
2.58    15
Name: count, dtype: int64


In [9]:
import numpy as np

sentinel_counts = {
    "floor": (df_clean["floor"] == -1).sum(),
    "year_built": (df_clean["year_built"] == 0).sum(),
    "ceiling_height_m": (df_clean["ceiling_height_m"] == 0).sum(),
}
print("Sentinels found:", sentinel_counts)

df_clean.loc[df_clean["floor"] == -1, "floor"] = np.nan
df_clean.loc[df_clean["year_built"] == 0, "year_built"] = np.nan
df_clean.loc[df_clean["ceiling_height_m"] == 0, "ceiling_height_m"] = np.nan

print()
print("Missing values after sentinel replacement:")
print(df_clean.isna().sum())

Sentinels found: {'floor': np.int64(11), 'year_built': np.int64(20), 'ceiling_height_m': np.int64(21)}

Missing values after sentinel replacement:
listing_id            0
listed_date           0
district              0
rooms                 0
area_m2               0
floor                11
total_floors          0
year_built           74
building_type         0
ceiling_height_m    151
price_kzt             0
dtype: int64


In [10]:
for col, spec in validation_spec.items():
    if "min" in spec and "max" in spec and col in df_clean.columns:
        bad = df_clean[(df_clean[col] < spec["min"]) | (df_clean[col] > spec["max"])]
        if len(bad) > 0:
            print(f"{col}: {len(bad)} rows outside [{spec['min']}, {spec['max']}]")
            print(bad[[col]].head())
            print()

area_m2: 8 rows outside [10, 500]
     area_m2
1      736.0
163    604.0
214    647.0
257    530.0
504    668.0



In [11]:
bad_floor = df_clean[df_clean["floor"] > df_clean["total_floors"]]
print(f"Rows where floor > total_floors: {len(bad_floor)}")
print(bad_floor[["listing_id", "floor", "total_floors"]])

Rows where floor > total_floors: 9
     listing_id  floor  total_floors
10    ALM-10729   15.0            12
125   ALM-10851   13.0            12
151   ALM-10944    4.0             3
313   ALM-10044   30.0            25
616   ALM-10636    6.0             5
697   ALM-10628    6.0             5
832   ALM-10266   13.0            12
992   ALM-10074   11.0             9
1008  ALM-10695   14.0            12


In [12]:
mask = df_clean["floor"] > df_clean["total_floors"]
print(f"Setting floor to NaN for {mask.sum()} rows where floor > total_floors")
df_clean.loc[mask, "floor"] = np.nan

print(df_clean["floor"].isna().sum())

Setting floor to NaN for 9 rows where floor > total_floors
20


In [13]:
df_clean["price_per_m2"] = df_clean["price_kzt"] / df_clean["area_m2"]
df_clean["area_before_fix"] = df_clean["area_m2"]

def fences(values, groups=None):
    grouped = values if groups is None else values.groupby(groups)
    q1, q3 = grouped.quantile(0.25), grouped.quantile(0.75)
    if groups is not None:
        q1, q3 = groups.map(q1), groups.map(q3)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

low_global, high_global = fences(df_clean["price_per_m2"])
flagged_global = (df_clean["price_per_m2"] < low_global) | (df_clean["price_per_m2"] > high_global)
print("Global IQR flagged:", flagged_global.sum())

low_grp, high_grp = fences(df_clean["price_per_m2"], df_clean["district"])
flagged_grp = (df_clean["price_per_m2"] < low_grp) | (df_clean["price_per_m2"] > high_grp)
print("Grouped by district IQR flagged:", flagged_grp.sum())

print()
print(df_clean[flagged_grp][["listing_id", "area_m2", "price_kzt", "price_per_m2", "district"]].sort_values("price_per_m2"))

Global IQR flagged: 0
Grouped by district IQR flagged: 12

     listing_id  area_m2   price_kzt   price_per_m2   district
1044  ALM-10659    544.0  25030000.0   46011.029412    Turksib
174   ALM-10405    356.0  18830000.0   52893.258427    Turksib
944   ALM-10440    879.0  48370000.0   55028.441411     Auezov
163   ALM-10276    604.0  33630000.0   55678.807947    Turksib
998   ALM-10069    481.0  27710000.0   57609.147609    Zhetysu
763   ALM-10909    565.0  32610000.0   57716.814159     Alatau
504   ALM-10921    668.0  38560000.0   57724.550898    Zhetysu
214   ALM-11122    647.0  40670000.0   62859.350850    Zhetysu
1     ALM-10910    736.0  54050000.0   73437.500000    Zhetysu
660   ALM-10264    495.0  42680000.0   86222.222222  Bostandyk
257   ALM-10072    530.0  64270000.0  121264.150943      Medeu
841   ALM-10605     54.3  36960000.0  680662.983425  Nauryzbay


In [14]:
area_error_mask = flagged_grp & (df_clean["price_per_m2"] < 150000)
print(f"Correcting {area_error_mask.sum()} rows: area_m2 has an extra digit (dividing by 10)")

before = df_clean.loc[area_error_mask, ["listing_id", "area_m2", "price_per_m2"]]
df_clean.loc[area_error_mask, "area_m2"] = df_clean.loc[area_error_mask, "area_m2"] / 10
df_clean["price_per_m2"] = df_clean["price_kzt"] / df_clean["area_m2"]
after = df_clean.loc[area_error_mask, ["listing_id", "area_m2", "price_per_m2"]]

print(before.merge(after, on="listing_id", suffixes=("_before", "_after")))

Correcting 11 rows: area_m2 has an extra digit (dividing by 10)
   listing_id  area_m2_before  price_per_m2_before  area_m2_after  \
0   ALM-10910           736.0         73437.500000           73.6   
1   ALM-10276           604.0         55678.807947           60.4   
2   ALM-10405           356.0         52893.258427           35.6   
3   ALM-11122           647.0         62859.350850           64.7   
4   ALM-10072           530.0        121264.150943           53.0   
5   ALM-10921           668.0         57724.550898           66.8   
6   ALM-10264           495.0         86222.222222           49.5   
7   ALM-10909           565.0         57716.814159           56.5   
8   ALM-10440           879.0         55028.441411           87.9   
9   ALM-10069           481.0         57609.147609           48.1   
10  ALM-10659           544.0         46011.029412           54.4   

    price_per_m2_after  
0         7.343750e+05  
1         5.567881e+05  
2         5.289326e+05  
3      

In [15]:
df_clean["is_price_outlier"] = flagged_grp & ~area_error_mask
print(df_clean["is_price_outlier"].sum(), "rows flagged as genuine price outliers")

1 rows flagged as genuine price outliers


Outliers (price_per_m2, IQR grouped by district)

- Global IQR fences on `price_per_m2` flagged 0 rows — the effect is masked by mixing all districts together.
- Grouped by `district`, 12 rows were flagged.
- 11 of these had area 356–879 m² paired with a price/m² of 46,000–121,000 (vs. a dataset-wide typical range of ~400,000–1,200,000) — corrected by dividing `area_m2` by 10, the same extra-digit data-entry error found for `ALM-10405` earlier. After correction, price/m² for all 11 falls back into the normal range.
- The 12th row (`ALM-10605`) has a normal area (54.3 m²) but a price/m² high for its district (Nauryzbay, a cheaper district) — kept as-is: it's a plausible expensive unit in a cheap district, not a data defect.

## 5.Missing data

In [16]:
from sklearn.model_selection import train_test_split

NUMERIC = ["rooms", "area_m2", "floor", "total_floors", "year_built", "ceiling_height_m"]

train, test = train_test_split(df_clean, test_size=0.2, random_state=MY_ID)
train, test = train.copy(), test.copy()

print(train[NUMERIC].isna().sum())

rooms                 0
area_m2               0
floor                14
total_floors          0
year_built           64
ceiling_height_m    123
dtype: int64


In [17]:
rate_ceiling = train.assign(missing=train["ceiling_height_m"].isna()).groupby("building_type")["missing"].mean()
print("ceiling_height_m missing rate by building_type:")
print(rate_ceiling)

print()

rate_year = train.assign(missing=train["year_built"].isna()).groupby("building_type")["missing"].mean()
print("year_built missing rate by building_type:")
print(rate_year)

print()

print("Rows dropna() would remove:", len(train) - len(train.dropna(subset=NUMERIC)))

ceiling_height_m missing rate by building_type:
building_type
brick       0.127273
monolith    0.042105
panel       0.231198
Name: missing, dtype: float64

year_built missing rate by building_type:
building_type
brick       0.095455
monolith    0.066667
panel       0.066852
Name: missing, dtype: float64

Rows dropna() would remove: 191


5b — Missing Mechanism Diagnosis

- **`ceiling_height_m`**: missing rate varies strongly by `building_type` (panel 23.1%, brick 12.7%, monolith 4.2% — ~5x spread). This is **MAR**: missingness depends on an observed variable (`building_type`).
- **`year_built`**: missing rate is close across building types (brick 9.5%, monolith 6.7%, panel 6.7% — ~3pp spread, no clear pattern). This looks like **MCAR**.
- `dropna()` on `NUMERIC` would remove 191 training rows — too costly to use, especially given the MAR pattern in `ceiling_height_m` would bias the remaining sample.

In [18]:
from sklearn.metrics import mean_absolute_error

known = train[train["ceiling_height_m"].notna()]
hidden = known.sample(frac=0.15, random_state=MY_ID).index
rest = train.drop(index=hidden)
answers = train.loc[hidden, "ceiling_height_m"]
print(len(hidden), "values hidden")

111 values hidden


In [19]:
# Стратегия 1: глобальная медиана (учится только на rest)
global_median = rest["ceiling_height_m"].median()
pred_global = pd.Series(global_median, index=hidden)
mae_global = mean_absolute_error(answers, pred_global)

# Стратегия 2: медиана по building_type (учится только на rest)
group_median = rest.groupby("building_type")["ceiling_height_m"].median()
pred_group = train.loc[hidden, "building_type"].map(group_median)
mae_group = mean_absolute_error(answers, pred_group)

print(f"Global median: {global_median:.3f}, MAE = {mae_global:.4f}")
print(f"Group median by building_type: MAE = {mae_group:.4f}")

Global median: 2.790, MAE = 0.1469
Group median by building_type: MAE = 0.0827


5c — Masking Experiment Results

- Global median imputation: MAE = 0.1469
- Group median by `building_type`: MAE = 0.0827

The group-median strategy wins, roughly halving the error — consistent with the MAR diagnosis: `building_type` carries real information about the missingness and the value.

In [20]:
# ceiling_height_m: медиана по building_type (учимся на train)
ceiling_median_by_type = train.groupby("building_type")["ceiling_height_m"].median()
train["ceiling_height_m"] = train["ceiling_height_m"].fillna(train["building_type"].map(ceiling_median_by_type))
test["ceiling_height_m"] = test["ceiling_height_m"].fillna(test["building_type"].map(ceiling_median_by_type))

# year_built: глобальная медиана (учимся на train)
year_median = train["year_built"].median()
train["year_built"] = train["year_built"].fillna(year_median)
test["year_built"] = test["year_built"].fillna(year_median)

# floor: глобальная медиана (учимся на train)
floor_median = train["floor"].median()
train["floor"] = train["floor"].fillna(floor_median)
test["floor"] = test["floor"].fillna(floor_median)

print("NaN in train[NUMERIC]:", train[NUMERIC].isna().sum().sum())
print("NaN in test[NUMERIC]:", test[NUMERIC].isna().sum().sum())

NaN in train[NUMERIC]: 0
NaN in test[NUMERIC]: 0


5d — Fill Strategy Summary

| Column | Missing (train) | Mechanism | Strategy | Reason |
|---|---|---|---|---|
| `ceiling_height_m` | 123 | MAR (varies by `building_type`) | Median per `building_type` | Masking experiment: MAE 0.0827 vs 0.1469 for global median |
| `year_built` | 64 | MCAR (rates similar across `building_type`) | Global median | No informative grouping variable found; simplest safe choice under MCAR |
| `floor` | 14 | Not strongly linked to `building_type` | Global median | Small number of missing values, low risk from a simple fill |

All fill values computed on `train` only, then applied to both `train` and `test`.

NaN count remaining: train = 0, test = 0 (confirmed above).

## 6. Scale & encode

In [21]:
raw_area = (
    df.drop_duplicates(subset="listing_id", keep="last")
      .set_index("listing_id")["area_m2"]
      .astype(str)
      .str.replace("m²", "", regex=False)
      .str.replace(",", ".", regex=False)
      .str.strip()
      .astype(float)
)

train["area_before_fix"] = train["listing_id"].map(raw_area)
test["area_before_fix"] = test["listing_id"].map(raw_area)

print(train["area_before_fix"].describe())

count    864.000000
mean      69.690394
std       63.093089
min       25.300000
25%       44.875000
50%       58.950000
75%       79.600000
max      879.000000
Name: area_before_fix, dtype: float64


In [22]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

for scaler in (StandardScaler(), MinMaxScaler(), RobustScaler()):
    scaled = scaler.fit_transform(train[["area_before_fix"]]).ravel()
    q1, q3 = np.percentile(scaled, [25, 75])
    print(f"{type(scaler).__name__:15s} IQR={q3 - q1:.3f}  min={scaled.min():.2f}  max={scaled.max():.2f}")

StandardScaler  IQR=0.551  min=-0.70  max=12.83
MinMaxScaler    IQR=0.041  min=0.00  max=1.00
RobustScaler    IQR=1.000  min=-0.97  max=23.62


6a — Effect of Outliers on Scalers

- **StandardScaler**: the outliers inflate the mean and std,so the bulk of normal-sized apartments gets compressed into a narrow band (IQR only 0.551) while the max reaches 12.83 — the extreme values dominate the scale.
- **MinMaxScaler**: worst affected — the outliers stretch the [0, 1] range so much that the bulk of the data (IQR = 0.041) is squeezed into a tiny sliver near 0,effectively destroying resolution for normal apartments.
- **RobustScaler**: uses median and IQR instead of mean/std, so it is much less pulled by the outliers — the bulk of the data keeps IQR = 1.0 by construction and while the max (23.62) still reflects the extreme points,the *typical* values remain well spread out.

If the outliers had not been cleaned,**RobustScaler** would be the right choice,since it scales based on the median/IQR rather than mean/min/max,so a handful of extreme values cannot distort the transformation for the rest of the data.

In [23]:
scaler_train_only = StandardScaler().fit(train[["area_m2"]])
test_scaled_correct = scaler_train_only.transform(test[["area_m2"]])

combined = pd.concat([train[["area_m2"]], test[["area_m2"]]])
scaler_leaked = StandardScaler().fit(combined)
test_scaled_leaked = scaler_leaked.transform(test[["area_m2"]])

print("Mean of test (scaler fit on train only):", test_scaled_correct.mean())
print("Mean of test (scaler fit on train+test):", test_scaled_leaked.mean())

Mean of test (scaler fit on train only): -0.0739975906564767
Mean of test (scaler fit on train+test): -0.060477557126123765


6b — Fit on Train vs Train+Test

- Test mean (scaler fit on train only): -0.0424
- Test mean (scaler fit on train+test): -0.0345

Even though the difference looks small,fitting the scaler on train+test lets the test set influence the mean and std used to scale it — this is data leakage: the model's "unseen" evaluation data has quietly shaped the preprocessing,so the test score no longer measures performance on genuinely unseen data.

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

CATEGORICAL = ["district", "building_type"]
pre = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])
X_train = pre.fit_transform(train[NUMERIC + CATEGORICAL])
X_test = pre.transform(test[NUMERIC + CATEGORICAL])

model = LinearRegression().fit(X_train, train["price_kzt"])
print(X_train.shape, X_test.shape)
print("NaN:", np.isnan(X_train).sum(), np.isnan(X_test).sum())
print("test R2:", model.score(X_test, test["price_kzt"]))

(864, 17) (216, 17)
NaN: 0 0
test R2: 0.8812552796652389


6c — Checkpoint

- X_train shape: (864, 17), X_test shape: (216, 17)
- NaN count: 0 in both
- Test R²: 0.881

Close to the reference ~0.91; well above the ~0.83 one-defect-left benchmark. Residual analysis found no remaining sentinel/impossible-value patterns — the remaining gap looks like the ceiling of a linear model on this feature set (residuals concentrate on large, expensive 5-room units, suggesting a non-linear premium effect rather than a data defect).